In [1]:
import numpy as np
import torch

torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=6, suppress=True)

In [2]:
import numpy as np
import torch
import casadi as ca

torch.set_default_dtype(torch.float64)
np.set_printoptions(precision=6, suppress=True)

In [3]:
# Import your surrogate
from Surrogate_ODE_Model.glc_surrogate_casadi import make_glc_well_surrogate

from Surrogate_ODE_Model.glc_surrogate_torch import glc_surrogate_dx_torch_nostuck
BSW = 0.20
GOR = 0.05
PI  = 3.0e-6

glc_well = make_glc_well_surrogate(BSW=BSW, GOR=GOR, PI=PI)

# Wrap into a pure CasADi Function for dx only
y_sym = ca.MX.sym("y", 3)
u_sym = ca.MX.sym("u", 2)
dx_sym, out_sym = glc_well(y_sym, u_sym)

F_dx = ca.Function("F_dx", [y_sym, u_sym], [dx_sym], ["y","u"], ["dx"])
print("CasADi dx shape:", F_dx.size_out(0))

CasADi dx shape: (3, 1)


In [5]:
# Pick a "reasonable" point (avoid crazy values first)
y_np = np.array([3e6, 2e6, 8e4], dtype=float)   # [m_G_an, m_G_tb, m_L_tb]
u_np = np.array([0.5, 0.7], dtype=float)        # [u1, u2]

dx_ca = np.array(F_dx(y_np, u_np)).reshape(-1)

y_t  = torch.tensor(y_np)
u_t  = torch.tensor(u_np)
dx_th = glc_surrogate_dx_torch_nostuck(y_t, u_t, BSW=BSW, GOR=GOR, PI=PI).detach().cpu().numpy().reshape(-1)

print("dx CasADi:", dx_ca)
print("dx Torch :", dx_th)
print("abs err :", np.abs(dx_ca - dx_th))
print("max abs err:", np.max(np.abs(dx_ca - dx_th)))

dx CasADi: [-7.014000e-11  3.307767e+26 -3.307767e+26]
dx Torch : [-7.014000e-11  3.307767e+26 -3.307767e+26]
abs err : [0. 0. 0.]
max abs err: 0.0


In [6]:
rng = np.random.default_rng(0)

N = 200
# Random positive states (keep them reasonable)
Y = np.column_stack([
    rng.uniform(1e5, 1e7, size=N),   # m_G_an
    rng.uniform(1e5, 1e7, size=N),   # m_G_tb
    rng.uniform(1e3,  2e5, size=N),  # m_L_tb
]).astype(float)

U = np.column_stack([
    rng.uniform(0.05, 1.0, size=N),  # u1
    rng.uniform(0.05, 1.0, size=N),  # u2
]).astype(float)

dx_ca_all = np.zeros((N,3))
for k in range(N):
    dx_ca_all[k,:] = np.array(F_dx(Y[k], U[k])).reshape(-1)

Y_t = torch.tensor(Y)
U_t = torch.tensor(U)
dx_th_all = glc_surrogate_dx_torch(Y_t, U_t, BSW=BSW, GOR=GOR, PI=PI).detach().cpu().numpy()

err = dx_ca_all - dx_th_all
print("mean abs err:", np.mean(np.abs(err)))
print("max  abs err:", np.max(np.abs(err)))
print("max rel err:", np.max(np.abs(err) / (1e-12 + np.abs(dx_ca_all))))

NameError: name 'glc_surrogate_dx_torch' is not defined

In [6]:
k = np.argmax(np.max(np.abs(err), axis=1))
print("worst sample index:", k)
print("y:", Y[k])
print("u:", U[k])
print("dx CasADi:", dx_ca_all[k])
print("dx Torch :", dx_th_all[k])
print("abs err :", np.abs(err[k]))

worst sample index: 164
y: [7779342.762208  819317.651618   25258.542046]
u: [0.325349 0.166619]
dx CasADi: [-1.233714e-10  1.022528e+25 -1.022528e+25]
dx Torch : [-1.233714e-10  1.022528e+25 -1.022528e+25]
abs err : [0.000000e+00 1.229864e+13 1.229864e+13]
